# Mach-Zehnder Interferometer in Linear Gravity

This notebook demonstrates the computation of the interferometer phase for a
Mach-Zehnder (MZ) atom interferometer in a uniform gravitational field using
the Baker-Campbell-Hausdorff (BCH) expansion.

**Reference:** C. Ufrecht, *Theoretical approach to high-precision atom
interferometry*, PhD thesis, Universitat Ulm, 2019, Chapter 1.

## Setup

The MZ interferometer consists of three laser pulses at times $t_0$, $t_1 = t_0 + T$,
$t_2 = t_0 + 2T$, splitting and recombining the atomic wavefunction into two paths.
The overlap operator is:

$$\hat{U}_1^\dagger(t_d, t_i)\hat{U}_2(t_d, t_i) = e^{i\hat{\Phi}}$$

For a Hamiltonian $\hat{H} = \frac{\hat{p}^2}{2m} + mg\hat{x}$, the well-known result is:

$$\varphi = -kgT^2 \quad \text{(Eq. 1.114 in Ufrecht)}$$

In [ ]:
import sympy as sy
from sympy import Rational, I, symbols, simplify, expand, latex
from IPython.display import Latex, display, Markdown

from Interferometry import Hamiltonian, Pulse, U, Interferometer
from poly_operator import PolyOpEx, moyal_commutator, bchn, hbar

## 1. Symbolic MZ phase in linear gravity

Define the Hamiltonian and interferometer sequence symbolically.
The `Interferometer` class composes the time-evolution and laser-pulse
operators via BCH and returns the overlap exponent as a quadratic operator.

In [ ]:
# Symbolic parameters
m, g, k, T = symbols('m g k T', positive=True)
hbar_sym = symbols('hbar')

# Hamiltonian: H = p^2/(2m) + m*g*x
# Format: [a, b, c, d, e, f] for H = a*p^2 + b*p + c*(px+xp) + d*x + e + f*x^2
H = Hamiltonian([Rational(1, 2) / m, 0, 0, m * g, 0, 0])

# MZ sequence (operator ordering: left = first to act on ket)
# Upper arm: Pulse(k) -> evolve(T) -> Pulse(-k) -> evolve(T)
# Lower arm: evolve(T) -> Pulse(k) -> evolve(T) -> Pulse(-k)
upper = [Pulse(k), U(H, T), Pulse(-k), U(H, T)]
lower = [U(H, T), Pulse(k), U(H, T), Pulse(-k)]

# Compute overlap via BCH at order 4 (exact for quadratic H)
interf = Interferometer(upper, lower, BCHOrder=4)
result_dict, result_opex = interf.overlap()

# Display the overlap exponent
print("Overlap operator exponent: exp(i * Phi) where Phi =")
for name, label in [('p2', 'p^2'), ('p', 'p'), ('px_xp', 'px+xp'),
                      ('x', 'x'), ('x2', 'x^2'), ('const', '1')]:
    val = simplify(result_dict[name])
    if val != 0:
        display(Latex(f"$\\quad {label}: \\quad {latex(val)}$"))

### Verification

The overlap is a pure phase (all operator-valued terms vanish), confirming
the interferometer is **closed** in linear gravity. The exponent is
$-ikgT^2$, giving the interferometer phase:

$$\varphi_0 = -kgT^2 \quad \text{(Ufrecht Eq. 1.114)}$$

In [ ]:
# Extract and verify the phase
phase = simplify(result_dict['const'])
expected = I * k * g * T**2

ratio = simplify(phase.subs([(m, 1), (hbar_sym, 1)]) / expected)
display(Latex(f"Phase coefficient: ${latex(phase)}$"))
display(Latex(f"Ratio to $ikgT^2$: ${latex(ratio)}$"))

sign_str = '+' if ratio == 1 else '-' if ratio == -1 else 'UNEXPECTED'
print(f"\nPhase = {sign_str}kgT^2")
print("(This matches Ufrecht Eq. 1.114: phi_0 = -kgT^2)")

# Verify all operator terms vanish
print("\nOperator terms (should all be zero):")
for name in ['p2', 'p', 'px_xp', 'x', 'x2']:
    val = simplify(result_dict[name].subs([(m, 1), (hbar_sym, 1)]))
    print(f"  {name} = {val}")

## 2. Numerical verification against matrix exponentiation

As a cross-check, we build the overlap operator by direct matrix exponentiation
in a truncated Fock space and compare with the BCH result.

In [ ]:
import numpy as np
from scipy.linalg import expm

N_FOCK = 50; M_BLOCK = 20

# Fock space operators (hbar=1)
a_op = np.zeros((N_FOCK, N_FOCK), dtype=complex)
for i in range(N_FOCK - 1):
    a_op[i, i + 1] = np.sqrt(i + 1)
adag_op = a_op.T.copy()
x_mat = (a_op + adag_op) / np.sqrt(2)
p_mat = -1j * (a_op - adag_op) / np.sqrt(2)
I_mat = np.eye(N_FOCK, dtype=complex)

def opex_to_matrix(opex):
    """Convert OpEx to Fock-space matrix (m=1, hbar=1)."""
    a = complex(simplify(opex.a).subs([(m, 1), (hbar_sym, 1)]))
    b = complex(simplify(opex.b).subs([(m, 1), (hbar_sym, 1)]))
    c = complex(simplify(opex.c).subs([(m, 1), (hbar_sym, 1)]))
    d = complex(simplify(opex.d).subs([(m, 1), (hbar_sym, 1)]))
    e = complex(simplify(opex.e).subs([(m, 1), (hbar_sym, 1)]))
    f = complex(simplify(opex.f).subs([(m, 1), (hbar_sym, 1)]))
    return (a * p_mat @ p_mat + b * p_mat +
            c * (x_mat @ p_mat + p_mat @ x_mat) +
            d * x_mat + e * I_mat + f * x_mat @ x_mat)

# Numeric parameters (small for BCH convergence)
m_n, g_n, k_n, T_n = Rational(1), Rational(1, 100), Rational(1, 10), Rational(1, 50)

H_num = Hamiltonian([Rational(1, 2) / m_n, 0, 0, m_n * g_n, 0, 0])

upper_n = [Pulse(k_n), U(H_num, T_n), Pulse(-k_n), U(H_num, T_n)]
lower_n = [U(H_num, T_n), Pulse(k_n), U(H_num, T_n), Pulse(-k_n)]

interf_n = Interferometer(upper_n, lower_n, BCHOrder=8)
_, res_opex = interf_n.overlap()

# BCH overlap matrix
Z_mat = opex_to_matrix(res_opex)
expZ = expm(Z_mat)

# Direct matrix multiplication
H_mat = opex_to_matrix(H_num)
pulse_p = expm(1j * float(k_n) * x_mat)
pulse_m = expm(-1j * float(k_n) * x_mat)
ev = lambda t: expm(-1j * H_mat * float(t))

U_upper = ev(T_n) @ pulse_m @ ev(T_n) @ pulse_p
U_lower = pulse_m @ ev(T_n) @ pulse_p @ ev(T_n)
overlap_mat = U_lower.conj().T @ U_upper

# Compare
diff = np.linalg.norm((expZ - overlap_mat)[:M_BLOCK, :M_BLOCK])
ref = np.linalg.norm(overlap_mat[:M_BLOCK, :M_BLOCK])
rel_err = diff / ref

print(f"BCH overlap vs direct matrix: relative error = {rel_err:.2e}")
print(f"Result: {'PASS' if rel_err < 1e-8 else 'FAIL'}")

## 3. Phase decomposition (Ufrecht Eq. 1.86-1.89)

The total phase of the MZ interferometer decomposes into three contributions
(Eq. 1.86-1.89 in Ufrecht):

$$\hat{\phi}_1 = \frac{1}{\hbar}(\Delta\chi^p \hat{z} - \Delta\chi^z \hat{p}_z) + \varphi_g$$

where:
- **Position displacement:** $\Delta\chi^z = -\frac{\hbar}{m}\sum_n k_n^{(-)} t_n$
- **Momentum displacement:** $\Delta\chi^p = \hbar \sum_n k_n^{(-)}$
- **Gravitational phase:** $\varphi_g = -\frac{1}{2}g \sum_n k_n^{(-)} t_n^2$ (Eq. 1.88)
- **Kinetic phase:** $\varphi_k = \frac{\hbar}{2m}\sum_{j\geq 2}\sum_{n=1}^{j-1} k_j^{(-)} k_n^{(+)} (t_j - t_n)$ (Eq. 1.89)

For a standard MZ ($k^{(-)} = (k, -2k, k)$ at $t = (0, T, 2T)$), both displacements
vanish ($\Delta\chi = 0$, closed interferometer), and $\varphi_g = -kgT^2$.

Let's verify this decomposition numerically.

In [ ]:
# Ufrecht phase decomposition for the standard MZ
# Pulse sequence: k_n^(1) = (k, 0, k), k_n^(2) = (0, k, 0)
# -> k_n^(-) = k_n^(2) - k_n^(1) = (-k, k, -k)  ... wait, 
# Convention: k^(-) = k^(2) - k^(1) where (1)=upper, (2)=lower
# Upper: +k at t=0, -k at t=T  (pulses on upper arm)
# Lower: +k at t=T, -k at t=2T (pulses on lower arm)
# k_n^(1) = (k, -k, 0),  k_n^(2) = (0, k, -k)
# k_n^(-) = k_n^(2) - k_n^(1) = (-k, 2k, -k)
# k_n^(+) = k_n^(2) + k_n^(1) = (k, 0, -k)
# Times: t_n = (0, T, 2T)

k_minus = [-1, 2, -1]  # in units of k
k_plus  = [1, 0, -1]   # in units of k
t_n = [0, 1, 2]         # in units of T

# Position displacement: Delta_chi^z = -(hbar/m) * sum k_n^(-) * t_n
delta_chi_z = -sum(km * tn for km, tn in zip(k_minus, t_n))
print(f"Delta_chi^z (in units of hbar*k*T/m) = {delta_chi_z}")

# Momentum displacement: Delta_chi^p = hbar * sum k_n^(-)
delta_chi_p = sum(k_minus)
print(f"Delta_chi^p (in units of hbar*k) = {delta_chi_p}")

# Gravitational phase: phi_g = -(g/2) * sum k_n^(-) * t_n^2
phi_g = -Rational(1, 2) * sum(km * tn**2 for km, tn in zip(k_minus, t_n))
print(f"phi_g (in units of k*g*T^2) = {phi_g}")

# Kinetic phase: phi_k = (hbar/(2m)) * sum_{j>=2} sum_{n<j} k_j^(-) k_n^(+) (t_j-t_n)
phi_k = Rational(1, 2) * sum(
    k_minus[j] * k_plus[n] * (t_n[j] - t_n[n])
    for j in range(1, 3) for n in range(j)
)
print(f"phi_k (in units of hbar*k^2*T/m) = {phi_k}")

print(f"\nClosed interferometer: Delta_chi = 0? {delta_chi_z == 0 and delta_chi_p == 0}")
print(f"Phase = -kgT^2? phi_g = {phi_g} (expected -1 in units of kgT^2)")
print(f"Kinetic phase vanishes? phi_k = {phi_k}")

## 4. PolyOpEx mode: beyond quadratic Hamiltonians

The `PolyOpEx` class extends the algebra to polynomial operators of arbitrary
degree, using the Moyal bracket as the commutator. This enables BCH
computations with anharmonic perturbations (cubic, quartic potentials).

Here we demonstrate the same MZ calculation using `use_poly=True` and verify
it matches the OpEx result for the quadratic case.

In [ ]:
# Compare OpEx and PolyOpEx modes for the numeric MZ
interf_poly = Interferometer(upper_n, lower_n, BCHOrder=8, use_poly=True)
dic_poly, _ = interf_poly.overlap()

dic_opex, _ = interf_n.overlap()

print("Comparison of overlap terms (OpEx vs PolyOpEx):")
print(f"{'Term':<10} {'OpEx':>20} {'PolyOpEx':>20} {'Match':>8}")
print("-" * 60)
for name in ['p2', 'p', 'px_xp', 'x', 'const', 'x2']:
    v1 = complex(simplify(dic_opex[name]).subs(hbar_sym, 1))
    v2 = complex(simplify(dic_poly[name]).subs(hbar_sym, 1))
    match = abs(v1 - v2) < 1e-12
    print(f"{name:<10} {v1.imag:>20.12e} {v2.imag:>20.12e} {'OK' if match else 'FAIL':>8}")